### Create subset of all gravity mains that connect directly to a given outfall

In [1]:
# imports
import os
from nwtrace import NWTrace
import pandas as pd
import geopandas as gpd

network_path = "data/sewer_test.geojson"

multiple = True
upstream_only = True
downstream_only = False
verbose = True

sewer_id_field = 'Sewer Gravity Asset Identification'
upstream_field = 'Sewer Gravity Upstream Maintenance Hole'
downstream_field = 'Sewer Gravity Downstream Maintenance Hole'

outfall_file = 'data/Outfalls_ID.csv'
id_field = 'ASSET_ID'

# this string will be added to the front of the output filename
outputname_extra = "example2_"

output_dir = f"./out"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

In [2]:
outfalls = pd.read_csv(outfall_file)[id_field].tolist()

target_endpoints = outfalls
# Single target example
# target_endpoints = ["JP5428128690"]

Load layers from files (may take some time)

In [3]:
# lines
outfalls = pd.read_csv(outfall_file)[id_field].tolist()

Initialize a new NWTrace object and prepare hash tables for traversal

In [4]:
result = []

sewershed = NWTrace(
    network=network_path,
    id_field=sewer_id_field,
    upstream_field = upstream_field,
    downstream_field = downstream_field,
    verbose=verbose,
    output_dir=output_dir,
)

Loading network file: `data\sewer_test.geojson`...


### Adding an Additional Node Connection

In the sample data, there is a connection that is not recorded - where a junction intersects with the middle of a sewer line, resulting in more than one node that the sewer can connect to upstream ('from'). To fix this, we can manually add that connection back unto the tree from a table. 

In [5]:
# additional connections
fittings = gpd.read_file("data/additional_node.geojson") # load the addtional file
nodes_up = (fittings[["FACILITYID", "TO_FIXED"]]
            .dropna(subset=["FACILITYID", "TO_FIXED"]) # remove rows with None values
            .rename(columns={"FACILITYID": 'node_id', "TO_FIXED": 'segment_id'}) # rename to standard column names
            .to_dict(orient="records")) # convert to a list of dictionaries

# add the connection
sewershed.add_upstream_nodes(nodes_up)

Added 1 node-segment connection(s)
Created 0 new node(s)
Created 0 new segment(s).



### Adding an Additional Segment Connection

In the sample data, there is a connection that is not recorded - where a junction intersects with the middle of a sewer line, resulting in more than one node that the sewer can connect to upstream ('from'). To fix this, we can manually add that connection back unto the tree from a table.

In [6]:
# TODO: Adjust to be a smaller, sample dataset
catchbasin_leads = gpd.read_file("data/more/catchbasin_leads.gpkg")

new_segs = (catchbasin_leads[["FACILITYID", "UP_ASSET_ID", "DN_ASSET_ID"]]
            .rename(columns={"FACILITYID": 'segment_id', "UP_ASSET_ID": 'from', "DN_ASSET_ID": 'to'})
            .set_index('node_id').to_dict(orient="index"))

sewershed.add_segments(new_segs)

Added 67943 node-segment connection(s)
Created 172726 new node(s)
Created 120788 new segment(s).



Run the search on the completed sewershed object

In [ ]:
if multiple == False:
    result = sewershed.trace_sewershed(
        target_endpoints[0], 
        upstream_only=upstream_only, 
        downstream_only=downstream_only
    )
else:
    result = sewershed.trace_sewersheds(
        target_endpoints, 
        upstream_only=upstream_only, 
        downstream_only=downstream_only, 
    )

Output the final connections to a CSV file, which can be added to GIS software

In [ ]:
# Convert to dataframe and output as a csv
out_df = pd.DataFrame.from_dict(result)
out_df.to_csv(f'{output_dir}/{outputname_extra}catchment_{"singledir" if upstream_only or downstream_only else "multidir"}{"_ups" if upstream_only and not downstream_only else ""}{"_dwns" if downstream_only and not upstream_only else ""}_{target_endpoints[0] if not multiple else outfall_file.replace('/', '_').replace('.', '_')}.csv')